In [6]:
import os
import openai
from dotenv import load_dotenv
from openai.types.chat import ChatCompletionMessage
import json

load_dotenv()

client = openai.OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("API_BASE_URL"), 
)

messages = []

def get_weather(city):
	return '33 degrees celcius.'

FUNCTION_MAP = {
	'get_weather': get_weather
}

TOOLS = [
	{
		"type": "function",
		"function": {
			"name": "get_weather",
			"description": "A function to get the weather of a city",
			"parameters": {
				"type": "object",
				"properties": {
					"city": {
						"type": "string",
						"description": "The name of the city to get the weather of."
					}
				},
				"required": ["city"],
			}
		}
	}
]

def process_ai_response(message: ChatCompletionMessage):
	if message.tool_calls:
		messages.append({
			"role": "assistant", 
			"content": message.content or "", 
			"tool_calls": [
				{
					"id": tool_call.id,
					"type": "function",
					"function": {
						"name": tool_call.function.name,
						"arguments": tool_call.function.arguments
					}
				} for tool_call in message.tool_calls
			],
		})

		for tool_call in message.tool_calls:
			function_name = tool_call.function.name
			arguments = tool_call.function.arguments

			print(f"Calling function: {function_name} with {arguments}")

			try:
				arguments = json.loads(arguments)
			except json.JSONDecodeError:
				arguments = {}

			function_to_run = FUNCTION_MAP.get(function_name)

			result = function_to_run(**arguments)

			print(f"Run {function_name} with args {arguments} fro a result of {result}")

			messages.append({
				"role": "tool",
				"tool_call_id": tool_call.id,
				"name": function_name,
				"content": result
			})

		call_ai()
	
	else:
		messages.append({ "role": "assistant", "content": message.content})
		print(f"AI: {message.content}")



def call_ai():
	response = client.chat.completions.create(
		model=os.getenv("API_MODEL", "deepseek-chat"),
		messages=messages,
		tools=TOOLS
	)
	process_ai_response(response.choices[0].message)


while True:
	message = input("Send a message to the LLM...")
	if message == "quit" or message == "q":
		break
	else:
		messages.append({"role": "user","content": message})
		print(f"user: {message}")
		call_ai()


user: меня зовут Игорь
AI: Приятно познакомиться, Игорь! Чем я могу вам помочь?
user: скажи какая погода в нижнем тагиле
Calling function: get_weather with {"city": "Нижний Тагил"}
Run get_weather with args {'city': 'Нижний Тагил'} fro a result of 33 degrees celcius.
AI: В Нижнем Тагиле сейчас **33°C** ☀️

Достаточно жаркая погода! Не забудьте пить воду и, если выходите на улицу, защищаться от солнца. 😊


In [7]:
messages

[{'role': 'user', 'content': 'меня зовут Игорь'},
 {'role': 'assistant',
  'content': 'Приятно познакомиться, Игорь! Чем я могу вам помочь?'},
 {'role': 'user', 'content': 'скажи какая погода в нижнем тагиле'},
 {'role': 'assistant',
  'content': '',
  'tool_calls': [{'id': 'call_00_TRngeV3lDkQ0uGRjOxG58108',
    'type': 'function',
    'function': {'name': 'get_weather',
     'arguments': '{"city": "Нижний Тагил"}'}}]},
 {'role': 'tool',
  'tool_call_id': 'call_00_TRngeV3lDkQ0uGRjOxG58108',
  'name': 'get_weather',
  'content': '33 degrees celcius.'},
 {'role': 'assistant',
  'content': 'В Нижнем Тагиле сейчас **33°C** ☀️\n\nДостаточно жаркая погода! Не забудьте пить воду и, если выходите на улицу, защищаться от солнца. 😊'}]